### Operational Lakebase — sessions DB on shared project

Creates the **`caspers_ops`** logical Postgres database inside the shared
`${CATALOG}-caspers` Lakebase Autoscaling project (provisioned by
`stages/lakebase_project.ipynb`), then creates the `sessions` and `messages`
tables that back the Operational Dashboard app's chat history.

The project creator (the deploying user) is automatically granted superuser
in every database in the project, so the DDL below runs cleanly without any
explicit GRANT.

In [ ]:
%pip install --upgrade "databricks-sdk>=0.81.0" "psycopg[binary]>=3.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re, sys
sys.path.append('../utils')
from lakebase_autoscale import get_or_create_postgres_database
import status

CATALOG = dbutils.widgets.get("CATALOG")

PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-caspers".lower())
PROJECT_RESOURCE_NAME = f"projects/{PROJECT_ID}"
BRANCH_PATH = f"{PROJECT_RESOURCE_NAME}/branches/production"
ENDPOINT_PATH = f"{BRANCH_PATH}/endpoints/primary"

# Lakebase Autoscale requires DNS-safe names (no underscores).
POSTGRES_DB = "caspers-ops"

print(f"CATALOG        = {CATALOG}")
print(f"PROJECT_ID     = {PROJECT_ID}")
print(f"ENDPOINT_PATH  = {ENDPOINT_PATH}")
print(f"POSTGRES_DB    = {POSTGRES_DB}")

##### Ensure the per-component Postgres database exists

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

db, created = get_or_create_postgres_database(
    w,
    project_id=PROJECT_ID,
    database_id=POSTGRES_DB,
)
if created:
    status.ok(f"Created Postgres database {POSTGRES_DB} in {PROJECT_ID}")
else:
    status.reuse(f"Reusing Postgres database {POSTGRES_DB} in {PROJECT_ID}")

##### Connect and create session tables

In [ ]:
import psycopg

ep = w.postgres.get_endpoint(name=ENDPOINT_PATH)
host = ep.status.hosts.host
user = w.current_user.me().user_name

cred = w.postgres.generate_database_credential(endpoint=ENDPOINT_PATH)

conn_str = (
    f"host={host} dbname={POSTGRES_DB} user={user} "
    f"password={cred.token} sslmode=require"
)

# Project creator has superuser — CREATE TABLE works without any GRANT.
STATEMENTS = [
    "CREATE TABLE IF NOT EXISTS sessions (id UUID PRIMARY KEY DEFAULT gen_random_uuid(), title TEXT NOT NULL DEFAULT 'New Session', created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(), updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW())",
    "CREATE TABLE IF NOT EXISTS messages (id UUID PRIMARY KEY DEFAULT gen_random_uuid(), session_id UUID NOT NULL REFERENCES sessions(id) ON DELETE CASCADE, role TEXT NOT NULL CHECK (role IN ('user', 'assistant')), content TEXT NOT NULL, created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(), documents_referenced JSONB DEFAULT '[]'::jsonb)",
    "CREATE INDEX IF NOT EXISTS idx_messages_session ON messages(session_id, created_at)",
    "CREATE INDEX IF NOT EXISTS idx_sessions_updated ON sessions(updated_at DESC)",
]

with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:
        for stmt in STATEMENTS:
            cur.execute(stmt)
    conn.commit()

status.ok(f"Tables ready in {PROJECT_ID}/{POSTGRES_DB}")
print(f"   Host: {host}")

In [ ]:
status.ok("Operational Lakebase stage complete")
print(f"   Project:     {PROJECT_RESOURCE_NAME}")
print(f"   Endpoint:    {ENDPOINT_PATH}")
print(f"   Postgres DB: {POSTGRES_DB}")
print(f"   Host:        {host}")